EVALUACION DE NAIVE BAYES - IMPLEMENTACION "MANUAL"

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import pandas as pd
import algoritmos_clasificacion as t
import importlib
importlib.reload(t)
# Hiperparámetros globales
m_values = [0.5, 1, 2]

def evaluar_naive_bayes(ruta_dataset):
    """
    Evalúa un Naive Bayes sobre el dataset en ruta_dataset.
    Devuelve un DataFrame con el resumen de métricas por cada valor de m.
    """
    resultados_bayes = {}

    # Cargar dataset
    S = pd.read_csv(ruta_dataset)

    # Features y target
    #X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
           #"amplitud_carrasco","amplitud_salto",
           #"dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
           #"dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom",
           #"vis_carrasco_prom","vis_salto_prom"]]
    y = S["dia_lluvioso"]
    X = S.drop(columns=["dia_lluvioso","fecha"])

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for m in m_values:
        print(f"\n=== Evaluando Naive Bayes con m={m} ===")
        
        # Crear modelo
        model = t.NaiveBayesClassifier(m=m)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Métricas en test
        acc_test = accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
        
        # Guardar resultados
        resultados_bayes[m] = {
            "cv_scores": cv_scores,
            "acc_test": acc_test,
            "f1_test": f1_test,
            "recall_test": recall_test,
            "cm": cm
        }

        # Prints intermedios
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)

    # Resumen final en tabla
    resumen = []
    for m, res in resultados_bayes.items():
        resumen.append({
            "m": m,
            "Accuracy CV (mean)": res["cv_scores"].mean(),
            "Accuracy CV (std)": res["cv_scores"].std(),
            "Accuracy test": res["acc_test"],
            "F1 test": res["f1_test"],
            "Recall test": res["recall_test"]
        })

    df_resumen = pd.DataFrame(resumen)
    print("\n=== RESUMEN NAIVE BAYES ===")
    print(df_resumen)

    return df_resumen, resultados_bayes

CON SCIKIT LEARN

In [4]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
import pandas as pd

# Hiperparámetros: al MultinomialNB no le pasamos m, así que solo dejamos alpha
alpha_values = [0.5, 1, 2, 5]

def evaluar_naive_bayes_sklearn(ruta_dataset):
    resultados_bayes = {}

    # Cargar dataset
    S = pd.read_csv(ruta_dataset)

    # Features y target
    X = S.drop(columns=["dia_lluvioso", "fecha"])
    y = S["dia_lluvioso"]

    # Convertir categorías a dummies (one-hot)
    X_enc = pd.get_dummies(X.astype(str))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.2, random_state=42, stratify=y
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for alpha in alpha_values:
        print(f"\n=== Evaluando MultinomialNB con alpha={alpha} ===")
        
        model = MultinomialNB(alpha=alpha)
        
        # Validación cruzada
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Métricas en test
        acc_test = accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, pos_label='si')
        recall_test = recall_score(y_test, y_pred, pos_label='si')
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])

        resultados_bayes[alpha] = {
            "cv_scores": cv_scores,
            "acc_test": acc_test,
            "f1_test": f1_test,
            "recall_test": recall_test,
            "cm": cm
        }

        # Prints intermedios
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)

    # Resumen final en tabla
    resumen = []
    for alpha, res in resultados_bayes.items():
        resumen.append({
            "alpha": alpha,
            "Accuracy CV (mean)": res["cv_scores"].mean(),
            "Accuracy CV (std)": res["cv_scores"].std(),
            "Accuracy test": res["acc_test"],
            "F1 test": res["f1_test"],
            "Recall test": res["recall_test"]
        })

    df_resumen = pd.DataFrame(resumen)
    print("\n=== RESUMEN NAIVE BAYES ===")
    print(df_resumen)

    return df_resumen, resultados_bayes




EVALUACIÓN DE LOS MODELOS

In [5]:
#modelo1_vent1 con implementacion manual
df_res1_1_m, res_bayes1_1_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent1.csv")
print(df_res1_1_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7217), np.float64(0.7217), np.float64(0.7214), np.float64(0.7183), np.float64(0.7208)]
Accuracy promedio CV: 0.7208 ± 0.0013
Accuracy en test: 0.7165
F1-score en test: 0.3552
Recall en test: 0.3522
Matriz de confusión:
[[ 317  583]
 [ 568 2592]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7217), np.float64(0.7217), np.float64(0.7214), np.float64(0.7183), np.float64(0.7208)]
Accuracy promedio CV: 0.7208 ± 0.0013
Accuracy en test: 0.7165
F1-score en test: 0.3552
Recall en test: 0.3522
Matriz de confusión:
[[ 317  583]
 [ 568 2592]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7217), np.float64(0.7217), np.float64(0.7214), np.float64(0.7183), np.float64(0.7208)]
Accuracy promedio CV: 0.7208 ± 0.0013
Accuracy en test: 0.7165
F1-score en test: 0.3552
Recall en test: 0.3522
Matriz de confusión:
[[ 317  583]
 [ 568 2592]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (

In [6]:
#modelo1_vent1 con implementacion de scikit
df_res1_1_s, res_bayes1_1_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent1.csv")
print(df_res1_1_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7214), np.float64(0.722), np.float64(0.7217), np.float64(0.7183), np.float64(0.7211)]
Accuracy promedio CV: 0.7209 ± 0.0013
Accuracy en test: 0.7165
F1-score en test: 0.3552
Recall en test: 0.3522
Matriz de confusión:
[[ 317  583]
 [ 568 2592]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7204), np.float64(0.7217), np.float64(0.7226), np.float64(0.7183), np.float64(0.7211)]
Accuracy promedio CV: 0.7208 ± 0.0014
Accuracy en test: 0.7165
F1-score en test: 0.3545
Recall en test: 0.3511
Matriz de confusión:
[[ 316  584]
 [ 567 2593]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7198), np.float64(0.722), np.float64(0.7226), np.float64(0.7183), np.float64(0.722)]
Accuracy promedio CV: 0.7209 ± 0.0016
Accuracy en test: 0.7163
F1-score en test: 0.3535
Recall en test: 0.3500
Matriz de confusión:
[[ 315  585]
 [ 567 2593]]

=== Evaluando MultinomialNB con al

In [7]:
#modelo1_vent2 con implementacion manual
df_res1_2_m, res_bayes1_2_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent2.csv")
print(df_res1_2_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7146), np.float64(0.7115), np.float64(0.7137), np.float64(0.7241), np.float64(0.7342)]
Accuracy promedio CV: 0.7196 ± 0.0085
Accuracy en test: 0.7185
F1-score en test: 0.3389
Recall en test: 0.3259
Matriz de confusión:
[[ 293  606]
 [ 537 2624]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7146), np.float64(0.7112), np.float64(0.7134), np.float64(0.7248), np.float64(0.7345)]
Accuracy promedio CV: 0.7197 ± 0.0088
Accuracy en test: 0.7187
F1-score en test: 0.3391
Recall en test: 0.3259
Matriz de confusión:
[[ 293  606]
 [ 536 2625]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7146), np.float64(0.7115), np.float64(0.7134), np.float64(0.7248), np.float64(0.7342)]
Accuracy promedio CV: 0.7197 ± 0.0086
Accuracy en test: 0.7190
F1-score en test: 0.3393
Recall en test: 0.3259
Matriz de confusión:
[[ 293  606]
 [ 535 2626]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (

In [8]:
#modelo1_vent2 con implementacion de scikit
df_res1_2_s, res_bayes1_2_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent2.csv")
print(df_res1_2_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7143), np.float64(0.7115), np.float64(0.7134), np.float64(0.7244), np.float64(0.7345)]
Accuracy promedio CV: 0.7196 ± 0.0087
Accuracy en test: 0.7192
F1-score en test: 0.3395
Recall en test: 0.3259
Matriz de confusión:
[[ 293  606]
 [ 534 2627]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7146), np.float64(0.7124), np.float64(0.7131), np.float64(0.7244), np.float64(0.7345)]
Accuracy promedio CV: 0.7198 ± 0.0085
Accuracy en test: 0.7195
F1-score en test: 0.3389
Recall en test: 0.3248
Matriz de confusión:
[[ 292  607]
 [ 532 2629]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7149), np.float64(0.7134), np.float64(0.7127), np.float64(0.7244), np.float64(0.7351)]
Accuracy promedio CV: 0.7201 ± 0.0086
Accuracy en test: 0.7197
F1-score en test: 0.3399
Recall en test: 0.3259
Matriz de confusión:
[[ 293  606]
 [ 532 2629]]

=== Evaluando MultinomialNB con

In [9]:
#modelo1_vent3 con implementacion manual
df_res1_3_m, res_bayes1_3_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent3.csv")
print(df_res1_3_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7368), np.float64(0.7365), np.float64(0.706), np.float64(0.7342), np.float64(0.741)]
Accuracy promedio CV: 0.7309 ± 0.0126
Accuracy en test: 0.7298
F1-score en test: 0.3044
Recall en test: 0.2670
Matriz de confusión:
[[ 240  659]
 [ 438 2723]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7365), np.float64(0.7368), np.float64(0.7063), np.float64(0.7339), np.float64(0.7413)]
Accuracy promedio CV: 0.7309 ± 0.0126
Accuracy en test: 0.7298
F1-score en test: 0.3044
Recall en test: 0.2670
Matriz de confusión:
[[ 240  659]
 [ 438 2723]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7365), np.float64(0.7365), np.float64(0.7063), np.float64(0.7336), np.float64(0.741)]
Accuracy promedio CV: 0.7308 ± 0.0125
Accuracy en test: 0.7296
F1-score en test: 0.3042
Recall en test: 0.2670
Matriz de confusión:
[[ 240  659]
 [ 439 2722]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (mea

In [10]:
#modelo1_vent3 con implementacion de scikit
df_res1_3_s, res_bayes1_3_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent3.csv")
print(df_res1_3_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7365), np.float64(0.7365), np.float64(0.7063), np.float64(0.7336), np.float64(0.7413)]
Accuracy promedio CV: 0.7308 ± 0.0125
Accuracy en test: 0.7298
F1-score en test: 0.3044
Recall en test: 0.2670
Matriz de confusión:
[[ 240  659]
 [ 438 2723]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7368), np.float64(0.7371), np.float64(0.7066), np.float64(0.7348), np.float64(0.7413)]
Accuracy promedio CV: 0.7313 ± 0.0125
Accuracy en test: 0.7300
F1-score en test: 0.3037
Recall en test: 0.2659
Matriz de confusión:
[[ 239  660]
 [ 436 2725]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7371), np.float64(0.7358), np.float64(0.7063), np.float64(0.7345), np.float64(0.741)]
Accuracy promedio CV: 0.7309 ± 0.0125
Accuracy en test: 0.7291
F1-score en test: 0.3011
Recall en test: 0.2636
Matriz de confusión:
[[ 237  662]
 [ 438 2723]]

=== Evaluando MultinomialNB con 

In [11]:
#modelo1_vent5 con implementacion manual
df_res1_5_m, res_bayes1_5_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent5.csv")
print(df_res1_5_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7432), np.float64(0.7407), np.float64(0.7512), np.float64(0.7561), np.float64(0.7715)]
Accuracy promedio CV: 0.7525 ± 0.0110
Accuracy en test: 0.7453
F1-score en test: 0.2453
Recall en test: 0.1869
Matriz de confusión:
[[ 168  731]
 [ 303 2858]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7432), np.float64(0.7407), np.float64(0.7512), np.float64(0.7558), np.float64(0.7715)]
Accuracy promedio CV: 0.7525 ± 0.0109
Accuracy en test: 0.7453
F1-score en test: 0.2453
Recall en test: 0.1869
Matriz de confusión:
[[ 168  731]
 [ 303 2858]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7432), np.float64(0.7401), np.float64(0.7515), np.float64(0.7555), np.float64(0.7715)]
Accuracy promedio CV: 0.7523 ± 0.0111
Accuracy en test: 0.7451
F1-score en test: 0.2451
Recall en test: 0.1869
Matriz de confusión:
[[ 168  731]
 [ 304 2857]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (

In [12]:
#modelo1_vent5 con implementacion de scikit
df_res1_5_s, res_bayes1_5_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent5.csv")
print(df_res1_5_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7435), np.float64(0.7404), np.float64(0.7512), np.float64(0.7558), np.float64(0.7715)]
Accuracy promedio CV: 0.7525 ± 0.0110
Accuracy en test: 0.7453
F1-score en test: 0.2453
Recall en test: 0.1869
Matriz de confusión:
[[ 168  731]
 [ 303 2858]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7435), np.float64(0.7401), np.float64(0.7508), np.float64(0.7558), np.float64(0.7718)]
Accuracy promedio CV: 0.7524 ± 0.0111
Accuracy en test: 0.7453
F1-score en test: 0.2453
Recall en test: 0.1869
Matriz de confusión:
[[ 168  731]
 [ 303 2858]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7445), np.float64(0.7413), np.float64(0.7512), np.float64(0.7561), np.float64(0.7724)]
Accuracy promedio CV: 0.7531 ± 0.0109
Accuracy en test: 0.7456
F1-score en test: 0.2443
Recall en test: 0.1858
Matriz de confusión:
[[ 167  732]
 [ 301 2860]]

=== Evaluando MultinomialNB con

In [13]:
#modelo1_vent7 con implementacion manual
df_res1_7_m, res_bayes1_7_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent7.csv")
print(df_res1_7_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7555), np.float64(0.769), np.float64(0.7752), np.float64(0.7721), np.float64(0.7598)]
Accuracy promedio CV: 0.7663 ± 0.0075
Accuracy en test: 0.7652
F1-score en test: 0.1903
Recall en test: 0.1246
Matriz de confusión:
[[ 112  787]
 [ 166 2994]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7558), np.float64(0.769), np.float64(0.7752), np.float64(0.7721), np.float64(0.7598)]
Accuracy promedio CV: 0.7664 ± 0.0074
Accuracy en test: 0.7652
F1-score en test: 0.1903
Recall en test: 0.1246
Matriz de confusión:
[[ 112  787]
 [ 166 2994]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7552), np.float64(0.769), np.float64(0.7752), np.float64(0.7721), np.float64(0.7592)]
Accuracy promedio CV: 0.7661 ± 0.0077
Accuracy en test: 0.7652
F1-score en test: 0.1903
Recall en test: 0.1246
Matriz de confusión:
[[ 112  787]
 [ 166 2994]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (mea

In [14]:
#modelo1_vent7 con implementacion de scikit
df_res1_7_s, res_bayes1_7_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent7.csv")
print(df_res1_7_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7555), np.float64(0.769), np.float64(0.7749), np.float64(0.7718), np.float64(0.7592)]
Accuracy promedio CV: 0.7661 ± 0.0075
Accuracy en test: 0.7652
F1-score en test: 0.1903
Recall en test: 0.1246
Matriz de confusión:
[[ 112  787]
 [ 166 2994]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7552), np.float64(0.7693), np.float64(0.7743), np.float64(0.7718), np.float64(0.7595)]
Accuracy promedio CV: 0.7660 ± 0.0074
Accuracy en test: 0.7657
F1-score en test: 0.1906
Recall en test: 0.1246
Matriz de confusión:
[[ 112  787]
 [ 164 2996]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7549), np.float64(0.7693), np.float64(0.7736), np.float64(0.7718), np.float64(0.7598)]
Accuracy promedio CV: 0.7659 ± 0.0073
Accuracy en test: 0.7657
F1-score en test: 0.1893
Recall en test: 0.1235
Matriz de confusión:
[[ 111  788]
 [ 163 2997]]

=== Evaluando MultinomialNB con 

In [15]:
#modelo1_vent10 con implementacion manual
df_res1_10_m, res_bayes1_10_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent10.csv")
print(df_res1_10_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.786), np.float64(0.7718), np.float64(0.7652), np.float64(0.7696), np.float64(0.7736)]
Accuracy promedio CV: 0.7732 ± 0.0069
Accuracy en test: 0.7736
F1-score en test: 0.1499
Recall en test: 0.0901
Matriz de confusión:
[[  81  818]
 [ 101 3059]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.786), np.float64(0.7718), np.float64(0.7649), np.float64(0.7696), np.float64(0.7736)]
Accuracy promedio CV: 0.7732 ± 0.0070
Accuracy en test: 0.7736
F1-score en test: 0.1499
Recall en test: 0.0901
Matriz de confusión:
[[  81  818]
 [ 101 3059]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7866), np.float64(0.7718), np.float64(0.7646), np.float64(0.7696), np.float64(0.7739)]
Accuracy promedio CV: 0.7733 ± 0.0073
Accuracy en test: 0.7736
F1-score en test: 0.1499
Recall en test: 0.0901
Matriz de confusión:
[[  81  818]
 [ 101 3059]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (me

In [16]:
#modelo1_vent10 con implementacion de scikit
df_res1_10_s, res_bayes1_10_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent10.csv")
print(df_res1_10_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7866), np.float64(0.7718), np.float64(0.7646), np.float64(0.7696), np.float64(0.7736)]
Accuracy promedio CV: 0.7732 ± 0.0073
Accuracy en test: 0.7736
F1-score en test: 0.1499
Recall en test: 0.0901
Matriz de confusión:
[[  81  818]
 [ 101 3059]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7866), np.float64(0.7721), np.float64(0.7643), np.float64(0.7699), np.float64(0.7736)]
Accuracy promedio CV: 0.7733 ± 0.0073
Accuracy en test: 0.7736
F1-score en test: 0.1499
Recall en test: 0.0901
Matriz de confusión:
[[  81  818]
 [ 101 3059]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7863), np.float64(0.7718), np.float64(0.7643), np.float64(0.7702), np.float64(0.7742)]
Accuracy promedio CV: 0.7733 ± 0.0072
Accuracy en test: 0.7733
F1-score en test: 0.1466
Recall en test: 0.0879
Matriz de confusión:
[[  79  820]
 [ 100 3060]]

=== Evaluando MultinomialNB con

In [17]:
#modelo2_vent1 con implementacion manual
df_res2_1_m, res_bayes2_1_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent1.csv")
print(df_res2_1_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7177), np.float64(0.7152), np.float64(0.7112), np.float64(0.7103), np.float64(0.7198)]
Accuracy promedio CV: 0.7148 ± 0.0037
Accuracy en test: 0.7076
F1-score en test: 0.3510
Recall en test: 0.3567
Matriz de confusión:
[[ 321  579]
 [ 608 2552]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7177), np.float64(0.7146), np.float64(0.7109), np.float64(0.7106), np.float64(0.7201)]
Accuracy promedio CV: 0.7148 ± 0.0037
Accuracy en test: 0.7079
F1-score en test: 0.3512
Recall en test: 0.3567
Matriz de confusión:
[[ 321  579]
 [ 607 2553]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7171), np.float64(0.7152), np.float64(0.7106), np.float64(0.7112), np.float64(0.7201)]
Accuracy promedio CV: 0.7148 ± 0.0036
Accuracy en test: 0.7081
F1-score en test: 0.3514
Recall en test: 0.3567
Matriz de confusión:
[[ 321  579]
 [ 606 2554]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (

In [18]:
#modelo2_vent1 con implementacion de scikit
df_res2_1_s, res_bayes2_1_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent1.csv")
print(df_res2_1_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7171), np.float64(0.7152), np.float64(0.7106), np.float64(0.7112), np.float64(0.7201)]
Accuracy promedio CV: 0.7148 ± 0.0036
Accuracy en test: 0.7081
F1-score en test: 0.3514
Recall en test: 0.3567
Matriz de confusión:
[[ 321  579]
 [ 606 2554]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7164), np.float64(0.7161), np.float64(0.7124), np.float64(0.7118), np.float64(0.7195)]
Accuracy promedio CV: 0.7153 ± 0.0028
Accuracy en test: 0.7084
F1-score en test: 0.3516
Recall en test: 0.3567
Matriz de confusión:
[[ 321  579]
 [ 605 2555]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7177), np.float64(0.7167), np.float64(0.7121), np.float64(0.7106), np.float64(0.7198)]
Accuracy promedio CV: 0.7154 ± 0.0035
Accuracy en test: 0.7096
F1-score en test: 0.3540
Recall en test: 0.3589
Matriz de confusión:
[[ 323  577]
 [ 602 2558]]

=== Evaluando MultinomialNB con

In [19]:
#modelo2_vent2 con implementacion manual
df_res2_2_m, res_bayes2_2_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent2.csv")
print(df_res2_2_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7081), np.float64(0.714), np.float64(0.7171), np.float64(0.7229), np.float64(0.7296)]
Accuracy promedio CV: 0.7183 ± 0.0074
Accuracy en test: 0.7089
F1-score en test: 0.3207
Recall en test: 0.3103
Matriz de confusión:
[[ 279  620]
 [ 562 2599]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7078), np.float64(0.714), np.float64(0.7171), np.float64(0.7232), np.float64(0.7296)]
Accuracy promedio CV: 0.7183 ± 0.0075
Accuracy en test: 0.7089
F1-score en test: 0.3207
Recall en test: 0.3103
Matriz de confusión:
[[ 279  620]
 [ 562 2599]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7081), np.float64(0.7143), np.float64(0.7174), np.float64(0.7229), np.float64(0.7296)]
Accuracy promedio CV: 0.7185 ± 0.0073
Accuracy en test: 0.7084
F1-score en test: 0.3195
Recall en test: 0.3092
Matriz de confusión:
[[ 278  621]
 [ 563 2598]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (me

In [20]:
#modelo2_vent2 con implementacion de scikit
df_res2_2_s, res_bayes2_2_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent2.csv")
print(df_res2_2_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7084), np.float64(0.7146), np.float64(0.7174), np.float64(0.7238), np.float64(0.7293)]
Accuracy promedio CV: 0.7187 ± 0.0072
Accuracy en test: 0.7089
F1-score en test: 0.3199
Recall en test: 0.3092
Matriz de confusión:
[[ 278  621]
 [ 561 2600]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7081), np.float64(0.7149), np.float64(0.7158), np.float64(0.7238), np.float64(0.7296)]
Accuracy promedio CV: 0.7185 ± 0.0075
Accuracy en test: 0.7094
F1-score en test: 0.3187
Recall en test: 0.3070
Matriz de confusión:
[[ 276  623]
 [ 557 2604]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7094), np.float64(0.7152), np.float64(0.7164), np.float64(0.7238), np.float64(0.733)]
Accuracy promedio CV: 0.7196 ± 0.0081
Accuracy en test: 0.7099
F1-score en test: 0.3175
Recall en test: 0.3048
Matriz de confusión:
[[ 274  625]
 [ 553 2608]]

=== Evaluando MultinomialNB con 

In [21]:
#modelo2_vent3 con implementacion manual
df_res2_3_m, res_bayes2_3_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent3.csv")
print(df_res2_3_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7294), np.float64(0.7309), np.float64(0.7097), np.float64(0.7271), np.float64(0.7351)]
Accuracy promedio CV: 0.7264 ± 0.0088
Accuracy en test: 0.7219
F1-score en test: 0.3061
Recall en test: 0.2770
Matriz de confusión:
[[ 249  650]
 [ 479 2682]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7294), np.float64(0.7312), np.float64(0.7097), np.float64(0.7268), np.float64(0.7354)]
Accuracy promedio CV: 0.7265 ± 0.0089
Accuracy en test: 0.7224
F1-score en test: 0.3065
Recall en test: 0.2770
Matriz de confusión:
[[ 249  650]
 [ 477 2684]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7297), np.float64(0.7318), np.float64(0.7097), np.float64(0.7268), np.float64(0.7354)]
Accuracy promedio CV: 0.7267 ± 0.0090
Accuracy en test: 0.7224
F1-score en test: 0.3065
Recall en test: 0.2770
Matriz de confusión:
[[ 249  650]
 [ 477 2684]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (

In [22]:
#modelo2_vent3 con implementacion de scikit
df_res2_3_s, res_bayes2_3_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent3.csv")
print(df_res2_3_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.73), np.float64(0.7318), np.float64(0.7097), np.float64(0.7268), np.float64(0.7348)]
Accuracy promedio CV: 0.7266 ± 0.0089
Accuracy en test: 0.7227
F1-score en test: 0.3084
Recall en test: 0.2792
Matriz de confusión:
[[ 251  648]
 [ 478 2683]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7303), np.float64(0.7318), np.float64(0.7103), np.float64(0.7271), np.float64(0.7348)]
Accuracy promedio CV: 0.7269 ± 0.0087
Accuracy en test: 0.7236
F1-score en test: 0.3100
Recall en test: 0.2803
Matriz de confusión:
[[ 252  647]
 [ 475 2686]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7306), np.float64(0.7318), np.float64(0.7103), np.float64(0.7274), np.float64(0.7376)]
Accuracy promedio CV: 0.7276 ± 0.0092
Accuracy en test: 0.7251
F1-score en test: 0.3111
Recall en test: 0.2803
Matriz de confusión:
[[ 252  647]
 [ 469 2692]]

=== Evaluando MultinomialNB con a

In [23]:
#modelo2_vent5 con implementacion manual
df_res2_5_m, res_bayes2_5_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent5.csv")
print(df_res2_5_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7482), np.float64(0.7324), np.float64(0.7413), np.float64(0.7447), np.float64(0.7601)]
Accuracy promedio CV: 0.7453 ± 0.0091
Accuracy en test: 0.7424
F1-score en test: 0.2550
Recall en test: 0.1991
Matriz de confusión:
[[ 179  720]
 [ 326 2835]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7475), np.float64(0.7321), np.float64(0.7416), np.float64(0.745), np.float64(0.7601)]
Accuracy promedio CV: 0.7453 ± 0.0091
Accuracy en test: 0.7426
F1-score en test: 0.2562
Recall en test: 0.2002
Matriz de confusión:
[[ 180  719]
 [ 326 2835]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7466), np.float64(0.7324), np.float64(0.7425), np.float64(0.7453), np.float64(0.7601)]
Accuracy promedio CV: 0.7454 ± 0.0089
Accuracy en test: 0.7433
F1-score en test: 0.2578
Recall en test: 0.2013
Matriz de confusión:
[[ 181  718]
 [ 324 2837]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (m

In [24]:
#modelo2_vent5 con implementacion de scikit
df_res2_5_s, res_bayes2_5_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent5.csv")
print(df_res2_5_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7469), np.float64(0.7318), np.float64(0.7425), np.float64(0.7456), np.float64(0.761)]
Accuracy promedio CV: 0.7456 ± 0.0094
Accuracy en test: 0.7438
F1-score en test: 0.2593
Recall en test: 0.2024
Matriz de confusión:
[[ 182  717]
 [ 323 2838]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7472), np.float64(0.7327), np.float64(0.7428), np.float64(0.7459), np.float64(0.7613)]
Accuracy promedio CV: 0.7460 ± 0.0092
Accuracy en test: 0.7446
F1-score en test: 0.2619
Recall en test: 0.2047
Matriz de confusión:
[[ 184  715]
 [ 322 2839]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.746), np.float64(0.7321), np.float64(0.7431), np.float64(0.7481), np.float64(0.7622)]
Accuracy promedio CV: 0.7463 ± 0.0097
Accuracy en test: 0.7441
F1-score en test: 0.2573
Recall en test: 0.2002
Matriz de confusión:
[[ 180  719]
 [ 320 2841]]

=== Evaluando MultinomialNB con a

In [25]:
#modelo2_vent7 con implementacion manual
df_res2_7_m, res_bayes2_7_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent7.csv")
print(df_res2_7_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7524), np.float64(0.7647), np.float64(0.7702), np.float64(0.769), np.float64(0.761)]
Accuracy promedio CV: 0.7635 ± 0.0064
Accuracy en test: 0.7652
F1-score en test: 0.2038
Recall en test: 0.1357
Matriz de confusión:
[[ 122  777]
 [ 176 2984]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.7527), np.float64(0.7647), np.float64(0.7699), np.float64(0.769), np.float64(0.761)]
Accuracy promedio CV: 0.7635 ± 0.0063
Accuracy en test: 0.7652
F1-score en test: 0.2038
Recall en test: 0.1357
Matriz de confusión:
[[ 122  777]
 [ 176 2984]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.7527), np.float64(0.7647), np.float64(0.7706), np.float64(0.7687), np.float64(0.761)]
Accuracy promedio CV: 0.7635 ± 0.0063
Accuracy en test: 0.7652
F1-score en test: 0.2038
Recall en test: 0.1357
Matriz de confusión:
[[ 122  777]
 [ 176 2984]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (mean)

In [26]:
#modelo2_vent7 con implementacion de scikit
df_res2_7_s, res_bayes2_7_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent7.csv")
print(df_res2_7_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.7527), np.float64(0.7635), np.float64(0.7702), np.float64(0.7693), np.float64(0.761)]
Accuracy promedio CV: 0.7634 ± 0.0064
Accuracy en test: 0.7652
F1-score en test: 0.2038
Recall en test: 0.1357
Matriz de confusión:
[[ 122  777]
 [ 176 2984]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.7524), np.float64(0.7629), np.float64(0.7706), np.float64(0.7696), np.float64(0.7616)]
Accuracy promedio CV: 0.7634 ± 0.0066
Accuracy en test: 0.7650
F1-score en test: 0.2010
Recall en test: 0.1335
Matriz de confusión:
[[ 120  779]
 [ 175 2985]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7518), np.float64(0.7629), np.float64(0.7699), np.float64(0.7709), np.float64(0.7616)]
Accuracy promedio CV: 0.7634 ± 0.0069
Accuracy en test: 0.7645
F1-score en test: 0.1980
Recall en test: 0.1313
Matriz de confusión:
[[ 118  781]
 [ 175 2985]]

=== Evaluando MultinomialNB con 

In [27]:
#modelo2_vent10 con implementacion manual
df_res2_10_m, res_bayes2_10_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent10.csv")
print(df_res2_10_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [np.float64(0.7807), np.float64(0.7709), np.float64(0.7693), np.float64(0.7711), np.float64(0.7705)]
Accuracy promedio CV: 0.7725 ± 0.0042
Accuracy en test: 0.7704
F1-score en test: 0.1418
Recall en test: 0.0857
Matriz de confusión:
[[  77  822]
 [ 110 3050]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [np.float64(0.781), np.float64(0.7702), np.float64(0.7693), np.float64(0.7714), np.float64(0.7708)]
Accuracy promedio CV: 0.7725 ± 0.0043
Accuracy en test: 0.7704
F1-score en test: 0.1418
Recall en test: 0.0857
Matriz de confusión:
[[  77  822]
 [ 110 3050]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [np.float64(0.781), np.float64(0.7702), np.float64(0.7693), np.float64(0.7714), np.float64(0.7711)]
Accuracy promedio CV: 0.7726 ± 0.0043
Accuracy en test: 0.7704
F1-score en test: 0.1418
Recall en test: 0.0857
Matriz de confusión:
[[  77  822]
 [ 110 3050]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (me

In [28]:
#modelo2_vent10 con implementacion de scikit
df_res2_10_s, res_bayes2_10_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent10.csv")
print(df_res2_10_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [np.float64(0.781), np.float64(0.7706), np.float64(0.7693), np.float64(0.7714), np.float64(0.7711)]
Accuracy promedio CV: 0.7727 ± 0.0042
Accuracy en test: 0.7706
F1-score en test: 0.1435
Recall en test: 0.0868
Matriz de confusión:
[[  78  821]
 [ 110 3050]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [np.float64(0.781), np.float64(0.7699), np.float64(0.7683), np.float64(0.7711), np.float64(0.7714)]
Accuracy promedio CV: 0.7724 ± 0.0045
Accuracy en test: 0.7714
F1-score en test: 0.1439
Recall en test: 0.0868
Matriz de confusión:
[[  78  821]
 [ 107 3053]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [np.float64(0.7813), np.float64(0.7696), np.float64(0.768), np.float64(0.7711), np.float64(0.7711)]
Accuracy promedio CV: 0.7722 ± 0.0047
Accuracy en test: 0.7711
F1-score en test: 0.1422
Recall en test: 0.0857
Matriz de confusión:
[[  77  822]
 [ 107 3053]]

=== Evaluando MultinomialNB con al